In [2]:
#변환코드
import json
import random

# ── 원본 파일 경로만 바꿔서 재사용 가능 ──
CON_PATH = 'CON_AUGMENTED_150.json'
PEP_PATH = 'PEP_AUGMENTED_150.json'
RPT_PATH = 'RPT_AUGMENTED_150.json'

with open(CON_PATH, encoding='utf-8') as f:
    con_data = json.load(f)
with open(PEP_PATH, encoding='utf-8') as f:
    pep_data = json.load(f)
with open(RPT_PATH, encoding='utf-8') as f:
    rpt_data = json.load(f)

print(f"CON {len(con_data)}개 / PEP {len(pep_data)}개 / RPT {len(rpt_data)}개 로드")
def load_prompt(path):
    with open(path, encoding='utf-8') as f:
        return f.read().strip()

CON_SYSTEM = load_prompt('con_system.txt')
PEP_SYSTEM = load_prompt('pep_system.txt')
RPT_SYSTEM = load_prompt('rpt_system.txt')

def con_to_messages(item):
    user_payload = {
        "category": item["category"],
        "clause_text": item["clause_text"],
        "refs": item["refs"],
    }
    assistant_payload = {
        "판정": item["label"],
        "근거": item["근거"],
    }
    return {
        "messages": [
            {"role": "system", "content": CON_SYSTEM},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(assistant_payload, ensure_ascii=False)},
        ],
        "meta": {"id": item["id"], "task": "CON", "category": item["category"], "pattern": item["pattern"], "label": item["label"]},
    }


def pep_to_messages(item):
    inp = item["input"]
    user_payload = {
        "description": inp["description"],
        "rfp_excerpt": inp["rfp_excerpt"],
        "pep_excerpt": inp["pep_excerpt"],
    }
    assistant_payload = {
        "판정": item["output"]["판정"],
        "코멘트": item["output"]["코멘트"],
    }
    return {
        "messages": [
            {"role": "system", "content": PEP_SYSTEM},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(assistant_payload, ensure_ascii=False)},
        ],
        "meta": {"id": item["id"], "task": "PEP", "description_id": inp["description"]["id"], "판정": item["output"]["판정"]},
    }


def rpt_to_messages(item):
    inp = item["input"]
    user_payload = {
        "description": inp["description"],
        "PEP_excerpt": inp["PEP_excerpt"],
        "RPT_excerpt": inp["RPT_excerpt"],
    }
    assistant_payload = {
        "판정": item["output"]["판정"],
        "코멘트": item["output"]["코멘트"],
    }
    return {
        "messages": [
            {"role": "system", "content": RPT_SYSTEM},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(assistant_payload, ensure_ascii=False)},
        ],
        "meta": {"id": item["id"], "task": "RPT", "description_id": inp["description"]["id"], "판정": item["output"]["판정"]},
    }


con_sft = [con_to_messages(i) for i in con_data]
pep_sft = [pep_to_messages(i) for i in pep_data]
rpt_sft = [rpt_to_messages(i) for i in rpt_data]


def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


write_jsonl('kanana_con_sft.jsonl', con_sft)
write_jsonl('kanana_pep_sft.jsonl', pep_sft)
write_jsonl('kanana_rpt_sft.jsonl', rpt_sft)

all_sft = con_sft + pep_sft + rpt_sft
random.seed(42)
random.shuffle(all_sft)
write_jsonl('kanana_all_sft.jsonl', all_sft)

print(f"CON {len(con_sft)} / PEP {len(pep_sft)} / RPT {len(rpt_sft)} / ALL {len(all_sft)} 저장 완료")

CON 150개 / PEP 150개 / RPT 150개 로드
CON 150 / PEP 150 / RPT 150 / ALL 450 저장 완료


In [3]:
# 분할 코드
import json
import random
from collections import defaultdict

INPUT_PATH = 'kanana_all_sft.jsonl'   # kanana_con_sft.jsonl / pep / rpt 도 동일하게 사용 가능
VALID_RATIO = 0.1                      # 필요시 0.15 등으로 조절
SEED = 42

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def write_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def stratify_key(row):
    meta = row['meta']
    task = meta.get('task', 'UNKNOWN')
    label = meta.get('label') or meta.get('판정') or 'UNKNOWN'
    return (task, label)

rows = load_jsonl(INPUT_PATH)
print(f"전체 {len(rows)}개 로드")

groups = defaultdict(list)
for row in rows:
    groups[stratify_key(row)].append(row)

random.seed(SEED)
train_rows, valid_rows = [], []

for key, group in groups.items():
    random.shuffle(group)
    n_valid = max(1, round(len(group) * VALID_RATIO))  # 그룹이 작아도 valid에 최소 1개는 배정
    valid_rows.extend(group[:n_valid])
    train_rows.extend(group[n_valid:])
    print(f"  {key}: 전체 {len(group)} -> train {len(group)-n_valid} / valid {n_valid}")

random.shuffle(train_rows)
random.shuffle(valid_rows)

write_jsonl('kanana_train.jsonl', train_rows)
write_jsonl('kanana_valid.jsonl', valid_rows)

print(f"\n최종 train {len(train_rows)}개 / valid {len(valid_rows)}개 저장 완료")

전체 450개 로드
  ('CON', '일치'): 전체 60 -> train 54 / valid 6
  ('CON', '불일치'): 전체 66 -> train 59 / valid 7
  ('PEP', '불가'): 전체 62 -> train 56 / valid 6
  ('PEP', '검토'): 전체 45 -> train 41 / valid 4
  ('RPT', '검토'): 전체 46 -> train 41 / valid 5
  ('PEP', '충족'): 전체 43 -> train 39 / valid 4
  ('RPT', '충족'): 전체 30 -> train 27 / valid 3
  ('RPT', '불가'): 전체 73 -> train 66 / valid 7
  ('CON', '검토'): 전체 24 -> train 22 / valid 2
  ('RPT', '검토(확인필요)'): 전체 1 -> train 0 / valid 1

최종 train 405개 / valid 45개 저장 완료


In [7]:
from transformers import AutoTokenizer
import json
tok = AutoTokenizer.from_pretrained("kakaocorp/kanana-1.5-8b-instruct-2505")

def check(path):
    mx = over = n = 0
    for line in open(path, encoding='utf-8'):
        msgs = json.loads(line)["messages"]
        text = tok.apply_chat_template(msgs, tokenize=False)   # 먼저 문자열로 렌더
        L = len(tok(text, add_special_tokens=False)["input_ids"])  # 그 문자열을 토큰화
        mx = max(mx, L); over += (L > 8192); n += 1
    print(f"{path}: {n}건 / 최대 {mx}토큰 / 8192초과 {over}건")

for p in ["kanana_con_sft.jsonl","kanana_pep_sft.jsonl","kanana_rpt_sft.jsonl"]:
    check(p)

kanana_con_sft.jsonl: 150건 / 최대 2328토큰 / 8192초과 0건
kanana_pep_sft.jsonl: 150건 / 최대 3134토큰 / 8192초과 0건
kanana_rpt_sft.jsonl: 150건 / 최대 6593토큰 / 8192초과 0건
